# 1. 데이터 불러오기

In [17]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import scipy.stats as stats

In [66]:
df = pd.read_csv('./data_for_eda/telecom_encoding.csv')
df.head()

,id,gender,age,year,school,income,job,phone_usage_per_m,telecom_change_yn,mobile_bundle,...,jeonbuk,jeonnam,gyeongbuk,gyeongnam,jeju,sejong,skt,kt,lgu,mvno
0,10002,1,45,2017,4,0,0,29,0,0,...,0,0,0,0,0,0,1,0,0,0
1,10002,1,46,2018,4,0,0,35,0,1,...,0,0,0,0,0,0,1,0,0,0
2,10002,1,47,2019,4,0,0,58,0,1,...,0,0,0,0,0,0,1,0,0,0
3,10002,1,48,2020,4,0,0,37,1,1,...,0,0,0,0,0,0,0,1,0,0
4,10002,1,49,2021,4,0,0,32,0,1,...,0,0,0,0,0,0,0,1,0,0


# 2. 모델 학습

### 2-1. Logistic Regression

##### 2-1-1. 첫 번째 시도

In [ ]:
features = [
    'gender',
    'birth_year',
    'mar',
    'income',
    'job',
    'region',
    'phone_usage_per_m',
    'mobile_bundle',
    'telecom'
]

X = df[features]
y = df['telecom_change_yn']

In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

cat_cols = ['gender','mar','income','job','region','mobile_bundle','telecom']
num_cols = ['birth_year','phone_usage_per_m']

In [21]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
    ]
)

model = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('clf', LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contai

In [22]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:,1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

              precision    recall  f1-score   support

           0       0.69      0.95      0.80      4646
           1       0.56      0.13      0.21      2310

    accuracy                           0.68      6956
   macro avg       0.63      0.54      0.50      6956
weighted avg       0.65      0.68      0.60      6956

ROC-AUC: 0.6316806991258131


##### 2-1-2. 두 번째 시도

- 현재 AUC가 약 0.63 정도로 높은 수치가 아니기 때문에, feature engineering으로 보완
- 파생 변수
    - 나이: age (year - birth_year)
    - 기간 변수
    - 교호효과(interaction effect) 추가

In [23]:
# 파생변수
# 1. 나이
df['age'] = df['year'] - df['birth_year']

# 2. 기간 변수
df['first_year'] = df.groupby('id')['year'].transform('min')
df['tenure_obs'] = df['year'] - df['first_year']  # 0,1,2,...

# 3. 교호 효과: 어느 통신사에서 결합상품이 있느냐?
df['telecom_bundle'] = df['telecom'].astype(str) + "_" + df['mobile_bundle'].astype(str)

In [24]:
features = [
    'age',
    'gender',
    'mar',
    'income',
    'job',
    'region',
    'telecom',
    'mobile_bundle',
    'tenure_obs',
    'phone_usage_per_m',
    'telecom_bundle'
]

X = df[features]
y = df['telcom_changed']

In [25]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

cat_cols = [
    'gender','mar','income','job','region',
    'mobile_bundle','telecom', 'telecom_bundle'
]

num_cols = [
    'age','tenure_obs', 'phone_usage_per_m'
]

In [26]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
    ]
)

model = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('clf', LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contai

In [27]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:,1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

              precision    recall  f1-score   support

           0       0.69      0.94      0.80      4646
           1       0.56      0.16      0.25      2310

    accuracy                           0.68      6956
   macro avg       0.63      0.55      0.52      6956
weighted avg       0.65      0.68      0.61      6956

ROC-AUC: 0.6475964987803129


### 2-2. Support Vector Machine

- 일반화 성능은 안정적이지만, 낼 수 있는 최대 역치가 낮음 -> 다른 모델을 사용하는 것이 좋음

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score, classification_report

# 전처리: 수치 스케일링 + 범주 원핫
preprocess_svm = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ],
    remainder="drop"
)       # 열 종류별로 다른 전처리 진행

svm = LinearSVC(
    C=1.0,                         # C가 작을 수록 규제가 강해져서 더 보수적으로 학습
    class_weight="balanced",       # 클래스 불균형 있을 때 자동으로 가중치 부여
    random_state=42
)

# 전처리 + 모델 묶기
svm_model = Pipeline([
    ("preprocess", preprocess_svm),     # 전처리
    ("clf", svm)                        # 모델 학습
])

svm_model.fit(X_train, y_train)

# LinearSVC는 predict_proba 없음 → decision_function 사용
scores = svm_model.decision_function(X_test)
y_pred = svm_model.predict(X_test)

print(classification_report(y_test, y_pred))
print("SVM ROC-AUC:", roc_auc_score(y_test, scores))

              precision    recall  f1-score   support

           0       0.77      0.61      0.68      4716
           1       0.43      0.62      0.51      2241

    accuracy                           0.61      6957
   macro avg       0.60      0.62      0.60      6957
weighted avg       0.66      0.61      0.63      6957

SVM ROC-AUC: 0.6635983667021303


In [59]:
from sklearn.model_selection import GridSearchCV, GroupKFold

cv = GroupKFold(n_splits=3)

param_grid = {"clf__C": [0.01, 0.1, 0.5, 1, 2, 5, 10]}

gs = GridSearchCV(
    svm_model,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

gs.fit(X, y, groups=df["id"])   # groups

print("Best params:", gs.best_params_)
print("Best CV AUC:", gs.best_score_)

Fitting 3 folds for each of 7 candidates, totalling 21 fits
Best params: {'clf__C': 5}
Best CV AUC: 0.6537641390929326


In [60]:
from sklearn.metrics import roc_auc_score

best = gs.best_estimator_
best.fit(X_train, y_train)
scores = best.decision_function(X_test)
print("Test AUC:", roc_auc_score(y_test, scores))

Test AUC: 0.6635925002431742


### 2-3. Decision Tree

##### 2-3-1. 첫 번째 시도

In [63]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score

dt = DecisionTreeClassifier(
    random_state=42,
    class_weight="balanced",
    max_depth=6,
    min_samples_leaf=20,
    min_samples_split=40
)

dt_model = Pipeline([
    ("preprocess", preprocess),
    ("clf", dt)
])

dt_model.fit(X_train, y_train)

y_proba = dt_model.predict_proba(X_test)[:, 1]
print("DT ROC-AUC:", roc_auc_score(y_test, y_proba))

DT ROC-AUC: 0.6919508682170016


In [65]:
from sklearn.model_selection import GridSearchCV, GroupKFold

groups = df["id"]

param_grid = {
    "clf__max_depth": [3, 4, 5, 6, 7, 8, None],
    "clf__min_samples_leaf": [5, 10, 20, 30, 50],
    "clf__min_samples_split": [2, 10, 20, 40, 80]
}

cv = GroupKFold(n_splits=3)

gs = GridSearchCV(
    dt_model,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

gs.fit(X, y, groups=groups)

print("Best params:", gs.best_params_)
print("Best CV AUC:", gs.best_score_)

Fitting 3 folds for each of 175 candidates, totalling 525 fits
Best params: {'clf__max_depth': None, 'clf__min_samples_leaf': 50, 'clf__min_samples_split': 2}
Best CV AUC: 0.7004208668450681


### 2-4. Random Forest

##### 2-4-1. 첫 번째 시도

In [41]:
features = [
    'gender',
    'birth_year',
    'mar',
    'income',
    'job',
    'region',
    'phone_usage_per_m',
    'mobile_bundle',
    'telecom'
]

X = df[features]
y = df['telcom_changed']

In [42]:
# 랜덤 포레스트: id 기준 분할 필요
from sklearn.model_selection import GroupShuffleSplit
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df['id']))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

In [43]:
# 범주형, 수치형 구분
from sklearn.preprocessing import OneHotEncoder

cat_cols = ['gender','mar','income','job','region','mobile_bundle','telecom']
num_cols = ['birth_year','phone_usage_per_m']

preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)

In [44]:
# Random Forest 모델 돌리기
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=800,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample",
    min_samples_leaf=5
)

rf_model = Pipeline([
    ("preprocess", preprocess),
    ("clf", rf)
])

rf_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contai

In [46]:
y_pred = rf_model.predict(X_test)
y_proba = rf_model.predict_proba(X_test)[:,1]

print(classification_report(y_test, y_pred))
print("RF ROC-AUC:", roc_auc_score(y_test, y_proba))

              precision    recall  f1-score   support

           0       0.76      0.65      0.70      4716
           1       0.44      0.57      0.49      2241

    accuracy                           0.63      6957
   macro avg       0.60      0.61      0.60      6957
weighted avg       0.66      0.63      0.64      6957

RF ROC-AUC: 0.654465898652569


##### 2-4-2. 두 번째 시도
- 로지스틱 회귀분석 때 적용했던 파생변수들을 적용

In [47]:
# 파생변수
# 1. 나이
df['age'] = df['year'] - df['birth_year']

# 2. 기간 변수
df['first_year'] = df.groupby('id')['year'].transform('min')
df['tenure_obs'] = df['year'] - df['first_year']  # 0,1,2,...

# 3. 교호 효과: 어느 통신사에서 결합상품이 있느냐?
df['telecom_bundle'] = df['telecom'].astype(str) + "_" + df['mobile_bundle'].astype(str)

In [48]:
features = [
    'age',
    'gender',
    'mar',
    'income',
    'job',
    'region',
    'telecom',
    'mobile_bundle',
    'tenure_obs',
    'phone_usage_per_m',
    'telecom_bundle'
]

X = df[features]
y = df['telcom_changed']

In [50]:
from sklearn.model_selection import GroupShuffleSplit
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df['id']))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

In [51]:
from sklearn.preprocessing import OneHotEncoder

cat_cols = ['gender','mar','income','job','region','mobile_bundle','telecom', 'telecom_bundle']
num_cols = ['age','tenure_obs', 'phone_usage_per_m']

preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)

In [52]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=800,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample",
    min_samples_leaf=5
)

rf_model = Pipeline([
    ("preprocess", preprocess),
    ("clf", rf)
])

rf_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contai

In [40]:
y_pred = rf_model.predict(X_test)
y_proba = rf_model.predict_proba(X_test)[:,1]

print(classification_report(y_test, y_pred))
print("RF ROC-AUC:", roc_auc_score(y_test, y_proba))

              precision    recall  f1-score   support

           0       0.80      0.65      0.72      4716
           1       0.47      0.67      0.55      2241

    accuracy                           0.65      6957
   macro avg       0.64      0.66      0.64      6957
weighted avg       0.70      0.65      0.66      6957

RF ROC-AUC: 0.7272995005183301


In [ ]:
df = pd.read_csv("/mnt/data/tel_preprocessed_ver1.csv")

features = [
    'gender','birth_year','mar','income','job','region',
    'phone_usage_per_m','mobile_bundle','telecom','year'
]

X = df[features]
y = df['telcom_changed']   # 타겟
groups = df['id']          # 그룹(누수 방지용)

##### 2-4-3. 세 번째 시도

In [ ]:
features = [
    'gender','birth_year','mar','income','job','region',
    'phone_usage_per_m','mobile_bundle','telecom','year'
]

X = df[features]
y = df['telcom_changed']   # 타겟
groups = df['id']          # 그룹(누수 방지용)

In [ ]:
cat_cols = ['gender','mar','income','job','region','mobile_bundle','telecom']
num_cols = ['birth_year','phone_usage_per_m','year']

In [ ]:
from sklearn.compose import ColumnTransformer
preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ],
    remainder="drop"
)

In [ ]:
rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample"
)

In [ ]:
pipe = Pipeline([
    ("preprocess", preprocess),
    ("clf", rf)
])

In [ ]:
from sklearn.model_selection import GroupKFold, RandomizedSearchCV

cv = GroupKFold(n_splits=5)

param_dist_fast = {
    "clf__n_estimators": [200, 400, 600],   # 1200 제거
    "clf__max_depth": [6, 8, 10, 12],       # None 제거(깊은 트리 느림)
    "clf__min_samples_leaf": [3, 5, 10],
    "clf__max_features": ["sqrt", 0.6],
}

rs_fast = RandomizedSearchCV(
    pipe,
    param_distributions=param_dist_fast,
    n_iter=12,               # 30 -> 12
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=2,               # 진행 로그 더 자세히
    random_state=42
)

rs_fast.fit(X, y, groups=groups)

print("Best CV AUC:", rs_fast.best_score_)
print("Best params:", rs_fast.best_params_)

NameError: name 'pipe' is not defined